# Project Assignment: Short Video Recommender System (KuaiRec)

Dataset Source: [Kuairec](https://kuairec.com/)

Arxiv Paper: [KuaiRec: A Fully-observed Dataset and Insights for Evaluating Recommender Systems](https://arxiv.org/pdf/2202.10842)

## Cosine Similarity Model
### Overview
This model is a naive approach to the recommendation problem.

### Description
Cosine similarity is a metric used to measure how similar two vectors are. Users and items can be represented as vectors in a multi-dimensional space, and cosine similarity can be used to find the most similar items to a given item or the most similar users to a given user.

Unlike collaborative filtering approaches that rely on user-item interaction patterns, this method leverages explicit item and user features, making it suitable even when interaction data is sparse.

### Evaluation

The system is evaluated as a binary classification problem. Given a set of recommended items, we assess whether the user genuinely liked the item. This is determined by checking if the user watched more than a defined threshold percentage (e.g., 70%) of the video.

For each recommendation, the label is:
- **Positive (1):** if the user watched more than the threshold (for example, watch_ratio > 0.7)
- **Negative (0):** otherwise

### Conclusion
This model is very simple, yet its performance is highly sensitive to data sparsity.

On the denser `small_matrix`, the results are generally better because it's easier to find meaningful user similarities and verify user-item interactions. However, on the sparser matrix `big_matrix`, performance drops significantly. This decline isn't necessarily due to poor recommendations, but rather to limitations in the evaluation process itself: since we can only validate a recommendation if the user actually interacted with the item in the test set, many relevant suggestions go unrecognized.

As a result, the model may appear to perform worse than it truly does. 

One way to solve this problem is to use `big_matrix` for the "training" and `small_matrix` for label verification.


### Possible extensions

This model can be improved with:
- **More features:** Adding more video features (kuareic_caption_category) and user features (social_network).
- **History-based recommendations:** Prioritizing items similar to the user's watch history.
- **Friend weighting:** Weighting the recommendations based on friendship connections.
- **Popularity scoring:** Incorporating item popularity into the item feature vector. More popular items may have a higher chance of being watched.


## Dataset import

The server is down, please download from the Google Drive in the given link.

In [ ]:
!wget https://nas.chongminggao.top:4430/datasets/KuaiRec.zip --no-check-certificate
!unzip KuaiRec.zip

--2025-05-09 17:20:44--  https://nas.chongminggao.top:4430/datasets/KuaiRec.zip
Resolving nas.chongminggao.top (nas.chongminggao.top)... 

In [2]:
# Misc
import numpy as np
import pandas as pd
from tqdm import tqdm
from utils import get_data_path, matrix_cleanup 

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split

# Model training
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score, precision_score, recall_score

# Plot metrics
import plotly.express as px
import plotly.graph_objects as go


# I get my dataset from a Kaggle input
DATA_PATH = get_data_path()

DATA_PATH

'/home/tofeha/ING2/ING2/REMA1/FinalProject_2025_aziz.zeghal/models/../KuaiRec 2.0/data'

# Step 1: Load the dataset

## Small matrix

This table has a density of 99.6%. This means that 99.6% of the entries in the matrix are non-zero, indicating that most users have interacted with most items.

In [3]:
small_matrix = pd.read_csv(f"{DATA_PATH}/small_matrix.csv")

small_matrix = matrix_cleanup(small_matrix)


In [4]:
small_matrix.head(3)

,user_id,video_id,play_duration,video_duration,timestamp,watch_ratio
0,14,148,4381,6067,2020-07-04 21:27:48.378000021,0.722103
1,14,183,11635,6100,2020-07-04 21:28:00.056999922,1.907377
2,14,3649,22422,10867,2020-07-04 21:29:09.479000092,2.063311


## Big matrix

This table has a density of 16.3%. We will use this matrix for our training and testing.

It contains more interactions with the same users/items of the small matrix. We do not need to substract the small matrix.

In [5]:
big_matrix = pd.read_csv(f"{DATA_PATH}/big_matrix.csv")

big_matrix = matrix_cleanup(big_matrix)


## Item category encoding

We have the caracteristics of the videos (author_id, video_type...) but this part requires less preprocessing.

For Content-based filtering, we need to use features of the videos (list of tags). We will use a simple one-hot encoding.

In [6]:
# No missing values for this data
item_categories = pd.read_csv(f"{DATA_PATH}/item_categories.csv")

## Item daily features

This dataset is also interesting for content-based filtering.

Mostly composed of textual data, we will use a TF-IDF vectorizer to encode the features of the videos.

In [7]:
item_daily_features = pd.read_csv(f"{DATA_PATH}/item_daily_features.csv", lineterminator='\n')
item_daily_features.fillna(-1, inplace=True)

## User features

In [8]:
user_features = pd.read_csv(f"{DATA_PATH}/user_features.csv", lineterminator='\n')
user_features.fillna(-1, inplace=True)

# Step 2: Feature Engineering

- Create meaningful features from interaction and metadata (e.g., content tags, user activity history)
- Build user-item interaction matrix
- Optionally extract time-based or popularity-based features

#### Item categories

In [9]:
# Use MultiLabelBinarizer to manage efficiently the feat column
mlb = MultiLabelBinarizer()

# Transform the feat column to a list (evaluate with python)
item_categories["feat"] = item_categories["feat"].apply(eval)

item_categories = pd.DataFrame(mlb.fit_transform(item_categories["feat"]), 
                  columns=mlb.classes_,
                  index=item_categories["video_id"])


item_categories.reset_index(drop=True, inplace=True)
item_categories[item_categories.columns] = item_categories[item_categories.columns].astype("int16")

item_categories.head(3)


,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


#### Item daily features

We take the oldest data point for a given video_id.

Depending on the complexity, you can choose the number of features.

In [10]:
TEXT_FEATURES = ["video_type", "upload_type"] # "visible_status"
INT_FEATURES = ['video_duration','video_width', 'video_height', 'music_id', 'video_tag_id','show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
                'play_duration', 'complete_play_cnt', 'complete_play_user_num', 'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
                'long_time_play_user_num', 'short_time_play_cnt', 'short_time_play_user_num', 'play_progress']
#  ['comment_stay_duration',
       # 'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       # 'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       # 'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       # 'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       # 'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       # 'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       # 'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       # 'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       # 'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       # 'cancel_collect_user_num']

In [11]:
# Keep the latest date for each video_id
item_daily_features = item_daily_features.loc[item_daily_features.groupby("video_id")["date"].idxmax()].reset_index(drop=True)

# One-hot str features
text_daily_features = item_daily_features[TEXT_FEATURES]
onehotter = OneHotEncoder(handle_unknown="ignore")
onehot_array = onehotter.fit_transform(text_daily_features).toarray()

# Convert to DataFrame
text_daily_features = pd.DataFrame(
    onehot_array,
    columns=onehotter.get_feature_names_out(TEXT_FEATURES),
    index=item_daily_features.index)

# Merge the one-hot encoded features back into the original DataFrame
item_daily_features = pd.concat([item_daily_features[INT_FEATURES], text_daily_features], axis=1)

In [12]:
# No IDs, because it is the index
item_features_map = pd.concat([item_daily_features, item_categories], axis=1)

# Column names should be str
item_features_map.columns = item_features_map.columns.map(str)

# We can keep all the columns
item_features_columns = item_features_map.columns.tolist()

### User features

In [13]:
user_features_columns = [
    "is_lowactive_period","is_live_streamer", "is_video_author",
    "onehot_feat0", "onehot_feat1", "onehot_feat2", "onehot_feat3",
    "onehot_feat4", "onehot_feat5", "onehot_feat6", "onehot_feat7",
    "onehot_feat8", "onehot_feat9", "onehot_feat10", "onehot_feat11", 
    "onehot_feat12", "onehot_feat13", "onehot_feat14", "onehot_feat15",
    "onehot_feat16", "onehot_feat17"
]
user_features_map = user_features[user_features_columns].copy()

user_features_map[user_features_map.columns] = user_features_map[user_features_map.columns].astype("int16")

In [14]:
# Index is the associated IDs for quick creation
display(user_features_map.head(3))
display(item_features_map.head(3))

,is_lowactive_period,is_live_streamer,is_video_author,onehot_feat0,onehot_feat1,onehot_feat2,onehot_feat3,onehot_feat4,onehot_feat5,onehot_feat6,...,onehot_feat8,onehot_feat9,onehot_feat10,onehot_feat11,onehot_feat12,onehot_feat13,onehot_feat14,onehot_feat15,onehot_feat16,onehot_feat17
0,0,0,0,0,1,17,638,2,0,1,...,184,6,3,0,0,0,0,0,0,0
1,0,0,0,0,3,25,1021,0,0,1,...,186,6,2,0,0,0,0,0,0,0
2,0,0,0,0,6,8,402,0,0,0,...,51,2,3,0,0,0,0,0,0,0


,video_duration,video_width,video_height,music_id,video_tag_id,show_cnt,show_user_num,play_cnt,play_user_num,play_duration,...,21,22,23,24,25,26,27,28,29,30
0,5966.0,720,1280,3350323409,8,3710,2649,2213,1635,19547072,...,0,0,0,0,0,0,0,0,0,0
1,-1.0,886,1015,1812462382,27,30,25,8,7,94830,...,0,0,0,0,0,0,1,0,0,0
2,8000.0,720,1280,0,9,72,59,17,16,212893,...,0,0,0,0,0,0,0,0,0,0


## Dataset preparation

In [15]:
INTERACTION_N = 20_000_000
# Either small_matrix or big_matrix
DATASET = big_matrix

In [16]:
interaction_matrix = DATASET.iloc[:INTERACTION_N][["user_id", "video_id", "watch_ratio"]].copy()

(user_ids, item_ids) = (interaction_matrix["user_id"].unique(), interaction_matrix["video_id"].unique())

# restrict the mappings to the unique user and item IDs
user_features_map = user_features_map.iloc[user_ids]
item_features_map = item_features_map.iloc[item_ids]


### Items similarity matrix

In [17]:
item_similarity = cosine_similarity(item_features_map[item_features_columns])

# Fill the diagonal with -1 to not recommend.
np.fill_diagonal(item_similarity, -1)

item_similarity = pd.DataFrame(
    item_similarity,
    index=item_features_map.index,
    columns=item_features_map.index
)

item_similarity[:3]


,3649,9598,5262,1963,8234,8228,6789,6812,183,169,...,9470,10429,7781,8,3817,4598,9476,2915,3246,3899
3649,-1.000000,0.999987,0.952640,0.999990,0.200630,0.999993,0.997370,0.999987,0.999988,0.999987,...,0.999987,0.999987,5.188157e-03,0.999987,0.999987,0.999987,4.849859e-03,0.999987,0.999987,0.999987
9598,0.999987,-1.000000,0.951049,1.000000,0.195547,0.999959,0.996981,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.104805e-08,1.000000,1.000000,1.000000,9.564814e-07,1.000000,1.000000,1.000000
5262,0.952640,0.951049,-1.000000,0.951242,0.488787,0.953798,0.972175,0.951061,0.951152,0.951049,...,0.951056,0.951052,3.089027e-01,0.951054,0.951049,0.951049,2.887084e-01,0.951050,0.951049,0.951049


### User similarity matrix

In [18]:
user_similarity = cosine_similarity(user_features_map[user_features_columns])

# Fill the diagonal with -1 to not recommend.
np.fill_diagonal(user_similarity, -1)

user_similarity = pd.DataFrame(
    user_similarity,
    index=user_features_map.index,
    columns=user_features_map.index
)

user_similarity[:3]


,0,1,2,3,4,5,6,7,8,9,...,7166,7167,7168,7169,7170,7171,7172,7173,7174,7175
0,-1.000000,0.994929,0.987952,0.898370,0.991432,0.984730,0.997970,0.999729,0.988632,0.964181,...,0.977139,0.988272,0.968627,0.999760,0.984066,0.909392,0.963448,0.997151,0.984270,0.987884
1,0.994929,-1.000000,0.998446,0.850243,0.983713,0.965007,0.986667,0.992628,0.996608,0.985518,...,0.993468,0.998523,0.988348,0.992878,0.996935,0.863069,0.985281,0.999264,0.996971,0.968105
2,0.987952,0.998446,-1.000000,0.820838,0.975514,0.950334,0.976383,0.984541,0.996756,0.992779,...,0.998042,0.999827,0.994796,0.984922,0.999619,0.834619,0.992810,0.996390,0.999574,0.953593


# Step 3: Model architecture

We will define the recommendation function based on a user_id.

The recommendation function will be applied to a subset of the dataset, to later verify if the user actually watched the video.

In [19]:
def similar(id : int, similarity_matrix: pd.DataFrame, n : int = 10) -> pd.DataFrame:
    """
    Get the most similar objects to the given id from the similarity matrix.

    Args:
        id (int): The id of the object to find similar objects for.
        similarity_matrix (pd.DataFrame): The similarity matrix (user or item).
        n (int): The number of similar objects to return.

    Returns:
        pd.Index: The indices of the most similar objects.
    """

    # Retrieve column of similarity for the id
    try:
        similars = similarity_matrix[id]
    except KeyError:
        print(f"The {id} was not found in the similarity matrix.")
        print(f"You can try with an id between {similarity_matrix.index[:3].values}")
        return None

    # Get the index of the most similar objects
    index_similars = similars.nlargest(10).index.sort_values()

    return index_similars
    


In [20]:
def recommend_for_user(user_id: int, interaction_matrix: pd.DataFrame, similarity_matrix: pd.DataFrame, n: int = 10):
    """
    Recommend n videos for the user_id

    Args:
        user_id (int): The id of the user to recommend videos for.
        interaction_matrix (pd.DataFrame): The visible interaction matrix containing user-video interactions.
        similarity_matrix (pd.DataFrame): The similarity matrix (user or item).
        n (int): The number of videos to recommend.
    """

    index_similars = similar(user_id, similarity_matrix, n)
    if index_similars is None:
        return None
    
    # Retrieve all interactions for similar users
    interactions_similar = interaction_matrix[interaction_matrix["user_id"].isin(index_similars)].copy()

    # Get the videos already seen by the target user
    user_seen_videos = interaction_matrix[interaction_matrix["user_id"] == user_id]["video_id"].unique()
    
    # Filter out already seen videos
    filtered = interactions_similar[~interactions_similar["video_id"].isin(user_seen_videos)]

    # Recommend top n videos with highest total watch_ratio
    recommended_videos = (
        filtered.groupby("video_id")["watch_ratio"]
        .sum()
        .sort_values(ascending=False)
        .head(n)
        .index
        .tolist()
    )

    return recommended_videos


In [21]:
train_df, test_df = interaction_matrix, small_matrix[["user_id", "video_id", "watch_ratio"]]

# Step 4: Recommendation

- Predict which videos are likely to be enjoyed by each user in the test set
- Generate a top-N ranked list of recommendations for each user

In [22]:
def generate_recommendation_df(
    interaction_matrix: pd.DataFrame,
    similarity_matrix: pd.DataFrame,
    users: list,
    recommend_func,
    k: int = 10,
) -> pd.DataFrame:
    """
    Generate a flat DataFrame of top-k recommendations for each user.

    Returns:
        DataFrame with columns: user_id, video_id, rank
    """
    rows = []

    for user_id in tqdm(users, desc="Generating recommendations"):
        video_ids = recommend_func(user_id, interaction_matrix, similarity_matrix, n=k)
        for rank, video_id in enumerate(video_ids or []):
            rows.append((user_id, video_id, rank, 1.0)) # No score for now

    return pd.DataFrame(rows, columns=["user_id", "video_id", "rank", "score"])


In [23]:
recs = generate_recommendation_df(
    interaction_matrix=train_df,
    similarity_matrix=user_similarity,
    users=train_df["user_id"].unique(),
    recommend_func=recommend_for_user,
    k=10
)

Generating recommendations: 100%|██████████| 7176/7176 [06:07<00:00, 19.53it/s]


In [24]:
def label_relevance(
    recommendations_df: pd.DataFrame,
    ground_truth_df: pd.DataFrame,
    watch_ratio_threshold: float = 0.5,
    k: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Label predictions with ground-truth relevance and return two pivoted DataFrames
    (for use in ranking metrics like NDCG).

    Args:
        recommendations_df: DataFrame with columns [user_id, video_id, rank]
        ground_truth_df: DataFrame with columns [user_id, video_id, watch_ratio]
        watch_ratio_threshold: Threshold above which an interaction is considered relevant
        k: Top-K items (used to pivot correctly)

    Returns:
        Tuple of (true_relevance_df, predicted_relevance_df)
    """
    # Step 1: Construct binary ground truth
    relevant_interactions = ground_truth_df[ground_truth_df["watch_ratio"] > watch_ratio_threshold]
    relevant_interactions = relevant_interactions[["user_id", "video_id"]].copy()
    relevant_interactions["relevance"] = 1

    # Step 2: Merge to label recommendations
    labeled_df = recommendations_df.merge(
        relevant_interactions,
        on=["user_id", "video_id"],
        how="left"
    ).fillna({"relevance": 0})

    # fillna incase values are missing
    true_relevance_df = labeled_df.pivot(index="user_id", columns="rank", values="relevance").fillna(0)
    predicted_score_df = labeled_df.pivot(index="user_id", columns="rank", values="score").fillna(1.0)

    # Ensure fixed size per user (pad if needed)
    for df in [true_relevance_df, predicted_score_df]:
        for i in range(k):
            if i not in df.columns:
                df[i] = 0
        df.sort_index(axis=1, inplace=True)

    return true_relevance_df, predicted_score_df


In [29]:
true, predicted = label_relevance(recs, test_df, watch_ratio_threshold=0.5)

In [30]:
predicted

rank,0,1,2,3,4,5,6,7,8,9
user_id,,,,,,,,,,
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...
7171,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7172,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7173,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [31]:

ndcg = ndcg_score(true, predicted, k=10)
precision = precision_score(true, predicted, average="macro")
recall = recall_score(true, predicted, average="macro")

In [32]:
ndcg, precision, recall

(np.float64(0.16761882391349905), 0.13761148272017837, 1.0)

In [122]:
similar(14, user_similarity)

Index([846, 885, 1161, 2057, 2526, 3821, 3987, 6014, 6388, 7117], dtype='int64')

In [62]:
recommend_for_user(14, train_df, user_similarity, n=10)

[3133, 1164, 586, 8366, 6993, 8506, 9157, 8628, 9850, 8732]

In [63]:
user_id = 14
recommendations = recommend_for_user(user_id, train_df, user_similarity, n=10)
print("Recommendations for user", user_id, ":", recommendations)

# To inspect what the user already saw:
seen = interaction_matrix[interaction_matrix["user_id"] == user_id]["video_id"].tolist()
print("Already watched:", seen)


Recommendations for user 14 : [3133, 1164, 586, 8366, 6993, 8506, 9157, 8628, 9850, 8732]
Already watched: [148, 183, 3649, 5262, 8234, 6789, 1963, 175, 1973, 171, 6803, 3634, 6787, 1951, 179, 5266, 5241, 6782, 6788, 8220, 6801, 3647, 6771, 9588, 186, 6812, 3684, 206, 211, 1988, 3672, 9595, 8242, 8248, 6829, 217, 9570, 139, 8160, 3669, 6846, 2007, 6839, 2000, 3654, 2008, 1898, 203, 5261, 256, 6854, 8289, 3702, 5326, 9660, 229, 5315, 262, 2040, 254, 2024, 5353, 6767, 8251, 8295, 5328, 8201, 9569, 2029, 223, 6865, 9653, 3706, 3630, 5331, 286, 2074, 2081, 3699, 9683, 1986, 2052, 285, 280, 5252, 8298, 5365, 9678, 275, 3719, 3586, 8212, 5367, 9592, 8319, 290, 145, 3737, 6904, 3722, 6749, 279, 147, 289, 5381, 1903, 9670, 8222, 296, 8316, 297, 2075, 2093, 9697, 2084, 3694, 3698, 5339, 2077, 258, 9645, 265, 2082, 1943, 5237, 3650, 5297, 8279, 3747, 307, 2113, 9659, 103, 5265, 8340, 5251, 6879, 3734, 288, 180, 5228, 5290, 8302, 8228, 8342, 6834, 5374, 9684, 5375, 3778, 6930, 9704, 340, 8323, 21

# Step 5: Evaluation

- Choose suitable metrics (e.g., Precision@K, Recall@K, MAP, NDCG)
- Evaluate performance and provide interpretations

In [64]:
def evaluate_user_recommender(
    y_train: pd.DataFrame,
    y_pred: pd.DataFrame,
    recommend_func,
    k: int = 10,
    watch_ratio_threshold: float = 0.5,
):
    """
    Evaluate a recommender system using Precision@K and Recall@K.

    Args:
        y_train (pd.DataFrame): The test set containing user-video interactions.
        y_pred (pd.DataFrame): The predicted interactions.
        recommend_func (callable): The recommendation function.
        k (int): The number of recommendations to consider.
        watch_ratio_threshold (float): The threshold for considering a video as relevant.

    Returns:
        dict
            Average precision and recall at K, and number of users evaluated.
    """
    user_ids = y_pred["user_id"].unique()
    precisions, recalls = [], []

    for user_id in tqdm(user_ids, desc="Evaluating users"):
        # Get recommended video IDs
        predicted_items = recommend_func(user_id, y_train, user_similarity, k)
        predicted_items = set(predicted_items or [])

        if not predicted_items:
            continue

        # Ground truth: videos with high watch_ratio in test set
        user_ground_truth = y_pred[y_pred["user_id"] == user_id]
        true_items = set(
            user_ground_truth[user_ground_truth["watch_ratio"] > watch_ratio_threshold]["video_id"]
        )

        if not true_items:
            continue

        hits = predicted_items & true_items

        precision = len(hits) / len(predicted_items)
        recall = len(hits) / len(true_items)

        precisions.append(precision)
        recalls.append(recall)

    avg_precision = sum(precisions) / len(precisions) if precisions else 0.0
    avg_recall = sum(recalls) / len(recalls) if recalls else 0.0

    return {
        f"precision@{k}": round(avg_precision, 4),
        f"recall@{k}": round(avg_recall, 4),
        "n_users_evaluated": len(precisions)
    }


In [65]:
metrics = evaluate_user_recommender(
    y_train=train_df,
    y_pred=test_df,
    recommend_func=recommend_for_user,
    k=10,
    watch_ratio_threshold=0.7
)

print(metrics)


Evaluating users: 100%|██████████| 1411/1411 [00:45<00:00, 30.86it/s]

{'precision@10': 0.6982, 'recall@10': 0.0197, 'n_users_evaluated': 1411}
